![Two data scientists working on a dashboard.](hr-image-small.png)

A common problem when creating models to generate business value from data is that the datasets can be so large that it can take days for the model to generate predictions. Ensuring that your dataset is stored as efficiently as possible is crucial for allowing these models to run on a more reasonable timescale without having to reduce the size of the dataset.

You've been hired by a major online data science training provider called *Training Data Ltd.* to clean up one of their largest customer datasets. This dataset will eventually be used to predict whether their students are looking for a new job or not, information that they will then use to direct them to prospective recruiters.

You've been given access to `customer_train.csv`, which is a subset of their entire customer dataset, so you can create a proof-of-concept of a much more efficient storage solution. The dataset contains anonymized student information, and whether they were looking for a new job or not during training:

| Column                   | Description                                                                      |
|------------------------- |--------------------------------------------------------------------------------- |
| `student_id`             | A unique ID for each student.                                                    |
| `city`                   | A code for the city the student lives in.                                        |
| `city_development_index` | A scaled development index for the city.                                         |
| `gender`                 | The student's gender.                                                            |
| `relevant_experience`    | An indicator of the student's work relevant experience.                          |
| `enrolled_university`    | The type of university course enrolled in (if any).                              |
| `education_level`        | The student's education level.                                                   |
| `major_discipline`       | The educational discipline of the student.                                       |
| `experience`             | The student's total work experience (in years).                                  |
| `company_size`           | The number of employees at the student's current employer.                       |
| `company_type`           | The type of company employing the student.                                       |
| `last_new_job`           | The number of years between the student's current and previous jobs.             |
| `training_hours`         | The number of hours of training completed.                                       |
| `job_change`             | An indicator of whether the student is looking for a new job (`1`) or not (`0`). |

In [2]:
# Import necessary libraries
import pandas as pd

# Load the dataset
ds_jobs = pd.read_csv("customer_train.csv")

# View the dataset
ds_jobs.head()

,student_id,city,city_development_index,gender,relevant_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,job_change
0,8949,city_103,0.920,Male,Has relevant experience,no_enrollment,Graduate,STEM,>20,NaN,NaN,1,36,1.0
1,29725,city_40,0.776,Male,No relevant experience,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,>4,47,0.0
2,11561,city_21,0.624,NaN,No relevant experience,Full time course,Graduate,STEM,5,NaN,NaN,never,83,0.0
3,33241,city_115,0.789,NaN,No relevant experience,NaN,Graduate,Business Degree,<1,NaN,Pvt Ltd,never,52,1.0
4,666,city_162,0.767,Male,Has relevant experience,no_enrollment,Masters,STEM,>20,50-99,Funded Startup,4,8,0.0


In [ ]:
# Create a copy of ds_jobs for transforming
ds_jobs_transformed = ds_jobs.copy()

def experience_to_numeric(exp_str):
    if pd.isna(exp_str):
        return 0
    exp_str = str(exp_str).strip()
    if exp_str == '<1':
        return 0
    elif exp_str == '>20':
        return 21
    else:
        try:
            return int(exp_str)
        except:
            return 0


ds_jobs_transformed['exp_numeric'] = ds_jobs_transformed['experience'].apply(experience_to_numeric)


large_companies = ['1000-4999', '5000-9999', '10000+']

ds_jobs_transformed = ds_jobs_transformed[
    (ds_jobs_transformed["exp_numeric"] >= 10) &  
    (ds_jobs_transformed["company_size"].isin(large_companies))
]

ds_jobs_transformed = ds_jobs_transformed.drop('exp_numeric', axis=1)

# Columns containing categories with only two factors must be stored as Booleans (bool).

for col in ds_jobs_transformed.columns: 
    unique_val = sorted(ds_jobs_transformed[col].dropna().unique())
    
    if len(unique_val) == 2 : 
        ds_jobs_transformed[col]  = ds_jobs_transformed[col].astype("category")

        mapping = {unique_val[0] : False , unique_val[1] : True}

        ds_jobs_transformed[col] = ds_jobs_transformed[col].map(mapping)
        ds_jobs_transformed[col] = ds_jobs_transformed[col].astype("bool")

# Columns containing integers only must be stored as 32-bit integers (int32).

for col in ds_jobs_transformed: 

    if ds_jobs_transformed[col].dtype == "int": 
        ds_jobs_transformed[col] = ds_jobs_transformed[col].astype("int32")

# Columns containing floats must be stored as 16-bit floats (float16).

for col in ds_jobs_transformed: 

    if ds_jobs_transformed[col].dtype == "float64" : 
        ds_jobs_transformed[col] = ds_jobs_transformed[col].astype("float16")


# Columns containing nominal categorical data must be stored as the category data type.

for col in ds_jobs_transformed.columns: 

    if ds_jobs_transformed[col].dtype == "object" : 
      unique = ds_jobs_transformed[col].dropna().unique()

      if len (unique ) < 20 : 
          ds_jobs_transformed[col] = ds_jobs_transformed[col].astype("category")


# Columns containing ordinal categorical data must be stored as ordered categories,
# and not mapped to numerical values, with an order that reflects the natural order of the column.

company_size_order = [
    '<10', '10/49', '50-99', '100-500', '500-999', '1000-4999', '5000-9999', '10000+'  
]

company_size_dtype = pd.CategoricalDtype(categories=company_size_order, ordered=True)
ds_jobs_transformed["company_size"] = ds_jobs_transformed["company_size"].astype(company_size_dtype)

education_order = [
    'Primary School', 'High School', 'Graduate', 'Masters', 'Phd'
]

education_dtype = pd.CategoricalDtype(categories=education_order, ordered=True)
ds_jobs_transformed["education_level"] = ds_jobs_transformed["education_level"].astype(education_dtype)

enrolled_university_order = [
    'no_enrollment',    
    'Part time course', 
    'Full time course'  
]
enrolled_university_dtype = pd.CategoricalDtype(categories=enrolled_university_order, ordered=True)
ds_jobs_transformed["enrolled_university"] = ds_jobs_transformed["enrolled_university"].astype(enrolled_university_dtype)

# Convert remaining columns to appropriate types
ds_jobs_transformed["city"] = ds_jobs_transformed['city'].astype("category")

experience_order = [
    '<1', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', 
    '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '>20'
]

experience_dtype = pd.CategoricalDtype(categories=experience_order, ordered=True) 
ds_jobs_transformed["experience"] = ds_jobs_transformed["experience"].astype(experience_dtype)

last_new_job_order = [
    'never', '1', '2', '3', '4', '>4'
]

last_new_job_dtype = pd.CategoricalDtype(categories=last_new_job_order, ordered=True)
ds_jobs_transformed["last_new_job"] = ds_jobs_transformed["last_new_job"].astype(last_new_job_dtype)
